# 🌩️ Weather Anomaly Detection Pipeline
**ERA5 fetch → Windy fetch → Isolation Forest → store scores**

## 0. Config

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import cdsapi
import xarray as xr
import requests
import psycopg2
from psycopg2.extras import execute_values
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import joblib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
warnings.filterwarnings('ignore')

# ── Spatial / temporal scope ───────────────────────────────────────────────────
REGION = {
    "lat_min": 40.5,  "lat_max": 55.5,   # Kazakhstan bounding box
    "lon_min": 50.0,  "lon_max": 87.5,
}
GRID_POINTS = [                            # Windy point-forecast inference grid
    {"lat": 51.18, "lon": 71.45},          # Astana
    {"lat": 43.25, "lon": 76.95},          # Almaty
    {"lat": 42.30, "lon": 69.60},          # Shymkent
]
ERA5_START   = "2024-01-01"
ERA5_END     = "2025-12-31"
ERA5_OUTFILE = "era5_raw.nc"

# ── Windy ──────────────────────────────────────────────────────────────────────
WINDY_KEY   = os.getenv("WINDY_API_KEY", "YOUR_KEY_HERE")
WINDY_MODEL = "gfs"   # 'gfs' | 'iconEu' | 'arome'
OPEN_METEO_URL = "https://api.windy.com/api/point-forecast/v2"

# ── TimescaleDB ────────────────────────────────────────────────────────────────
DB_CONFIG = {
    "host":     os.getenv("TSDB_HOST",     "localhost"),
    "port":     int(os.getenv("TSDB_PORT", "5432")),
    "dbname":   os.getenv("TSDB_DBNAME",   "weather_anomalies"),
    "user":     os.getenv("TSDB_USER",     "postgres"),
    "password": os.getenv("TSDB_PASSWORD", ""),
}

# ── Isolation Forest ──────────────────────────────────────────────────────────
IF_CONTAMINATION = 0.001   # expected anomaly fraction
IF_N_ESTIMATORS  = 200
MODEL_PATH       = "isolation_forest.joblib"
SCALER_PATH      = "scaler.joblib"

print("✅ Config loaded")

## 1. ERA5 — Fetch Historical Data

In [ ]:
import glob

# CDS API reads ~/.cdsapirc for credentials automatically
# File format:
#   url: https://cds.climate.copernicus.eu/api/v2
#   key: <UID>:<API-KEY>

c      = cdsapi.Client()
months = [f"{m:02d}" for m in range(1, 13)]
days   = [f"{d:02d}" for d in range(1, 32)]
times  = [f"{h:02d}:00" for h in range(0, 24, 3)]  # 3-hourly

# Fetch one file per year — avoids CDS 403 'request too large'
yearly_files = []
for year in range(int(ERA5_START[:4]), int(ERA5_END[:4]) + 1):
    outfile = f"era5_raw_{year}.nc"
    yearly_files.append(outfile)
    if os.path.exists(outfile):
        print(f"⏭️  {outfile} already exists, skipping")
        continue
    print(f"⬇️  Fetching ERA5 {year}...")
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable": [
                "10m_u_component_of_wind",
                "10m_v_component_of_wind",
                "total_precipitation",
                "2m_temperature",
            ],
            "year":  str(year),
            "month": months,
            "day":   days,
            "time":  times,
            "area":  [REGION["lat_max"], REGION["lon_min"],
                      REGION["lat_min"], REGION["lon_max"]],
            "format": "netcdf",
        },
        outfile,
    )
    print(f"✅ Saved {outfile}")

ERA5_OUTFILE = yearly_files  # passed as list to xr.open_mfdataset in next cell
print(f"\n📦 Files ready: {yearly_files}")

## 2. ERA5 — Preprocess into Feature DataFrame

In [ ]:
import zipfile

extracted = []
for zf in ERA5_OUTFILE:
    out_dir = zf.replace(".nc", "_extracted")
    os.makedirs(out_dir, exist_ok=True)
    with zipfile.ZipFile(zf, "r") as z:
        z.extractall(out_dir)
        extracted += [os.path.join(out_dir, f) for f in z.namelist() if f.endswith(".nc")]

ds = xr.concat([xr.open_dataset(f, engine="netcdf4") for f in extracted], dim="time")
print(ds)

In [ ]:
def preprocess_era5(ds: xr.Dataset) -> pd.DataFrame:
    def get_var(name):
        for i in range(ds.dims["time"]):
            arr = ds.isel(time=i)[name].values
            if not np.isnan(arr).all():
                return ds.isel(time=i)[name]
        raise ValueError(f"all time slices are NaN for {name}")

    lat = ds["latitude"].values
    lon = ds["longitude"].values
    lat_g, lon_g = np.meshgrid(lat, lon, indexing="ij")
    lat_flat = lat_g.ravel()
    lon_flat = lon_g.ravel()

    u10 = get_var("u10")
    v10 = get_var("v10")
    t2m = get_var("t2m")
    tp  = get_var("tp")

    chunks = []
    prev_tp   = None
    prev_hour = None

    for i in range(len(ds["valid_time"])):
        wu   = u10.isel(valid_time=i).values.ravel()
        wv   = v10.isel(valid_time=i).values.ravel()
        tk   = t2m.isel(valid_time=i).values.ravel()
        tpi  = tp.isel(valid_time=i).values.ravel()
        hour = pd.Timestamp(ds["valid_time"].values[i]).hour

        reset = (prev_hour is not None) and (hour in (1, 12))

        if prev_tp is None or reset:
            precip_3h = tpi.copy()
        else:
            precip_3h = np.clip(tpi - prev_tp, 0, None)
        prev_tp   = tpi.copy()
        prev_hour = hour

        chunks.append(pd.DataFrame({
            "time":       ds["valid_time"].values[i],
            "lat":        lat_flat,
            "lon":        lon_flat,
            "wind_u":     wu,
            "wind_v":     wv,
            "wind_speed": np.sqrt(wu**2 + wv**2),
            "precip_3h":  precip_3h,
            "temp_k":     tk,
        }))

    return pd.concat(chunks, ignore_index=True)

era5_df = preprocess_era5(ds)
print(f"ERA5 rows: {len(era5_df):,}")
era5_df.head()

In [ ]:
# Quick sanity check — distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
era5_df["wind_speed"].hist(bins=60, ax=axes[0], color="steelblue");  axes[0].set_title("vent (m/s)")
era5_df["precip_3h"].hist(bins=60, ax=axes[1], color="teal");        axes[1].set_title("pluie 3h (m)")
era5_df["temp_k"].hist(bins=60, ax=axes[2], color="tomato");         axes[2].set_title("temp (K)")
plt.tight_layout()
plt.show()

## 3. ERA5 — Insert into TimescaleDB

In [ ]:
def insert_era5(df: pd.DataFrame, batch_size: int = 2000):
    conn = psycopg2.connect(**DB_CONFIG)
    cur  = conn.cursor()

    sql = """
        INSERT INTO era5_observations (time, lat, lon, wind_u, wind_v, precip_3h, temp_k)
        VALUES %s
        ON CONFLICT (time, lat, lon) DO NOTHING
    """
    cols = ["time", "lat", "lon", "wind_u", "wind_v", "precip_3h", "temp_k"]
    for i in range(0, len(df), batch_size):
        batch = df.iloc[i:i+batch_size][cols].values.tolist()
        execute_values(cur, sql, batch)
        conn.commit()
        print(f"  inserted {min(i+batch_size, len(df))}/{len(df)}")

    cur.close()
    conn.close()
    print(f"✅ ERA5 insert done — {len(df):,} rows")

insert_era5(era5_df)

## 4. Windy — Live Point-Forecast Fetch

In [ ]:
def fetch_open_meteo(lat: float, lon: float) -> pd.DataFrame:
    resp = requests.get(OPEN_METEO_URL, params={
        "latitude":        lat,
        "longitude":       lon,
        "hourly":          ["wind_speed_10m", "precipitation", "temperature_2m"],
        "wind_speed_unit": "ms",
        "temperature_unit":"celsius",
        "timeformat":      "unixtime",
        "timezone":        "UTC",
        "forecast_days":   16,
    }, timeout=15)
    resp.raise_for_status()
    d = resp.json()["hourly"]

    df = pd.DataFrame({
        "time":       pd.to_datetime(d["time"], unit="s", utc=True),
        "wind_speed": np.array(d["wind_speed_10m"], dtype=float),
        "precip_3h":  np.array(d["precipitation"],  dtype=float) / 1000,
        "temp_k":     np.array(d["temperature_2m"], dtype=float) + 273.15,
    }).set_index("time")

    # resample to 3h: wind/temp → mean, precip → sum (accumulate over 3h)
    df = pd.DataFrame({
        "wind_speed": df["wind_speed"].resample("3h").mean(),
        "precip_3h":  df["precip_3h"].resample("3h").sum(),
        "temp_k":     df["temp_k"].resample("3h").mean(),
    }).reset_index()

    df["lat"]    = lat
    df["lon"]    = lon
    df["wind_u"] = df["wind_speed"]
    df["wind_v"] = 0.0

    return df


windy_df = pd.concat(
    [fetch_open_meteo(p["lat"], p["lon"]) for p in GRID_POINTS],
    ignore_index=True
).dropna(subset=FEATURES)

print(f"✅ Open-Meteo rows fetched: {len(windy_df)}")
windy_df.head()

In [ ]:
def insert_windy(df: pd.DataFrame, config: dict):
    conn = psycopg2.connect(**config)
    cur  = conn.cursor()

    records = [
        (row.time, row.lat, row.lon, row.model, row.wind_u, row.wind_v, row.precip_3h, row.temp_k)
        for row in df.itertuples()
    ]

    sql = """
        INSERT INTO windy_forecasts (time, lat, lon, model, wind_u, wind_v, precip_3h, temp_k)
        VALUES %s
        ON CONFLICT (time, lat, lon, model) DO UPDATE SET
            wind_u    = EXCLUDED.wind_u,
            wind_v    = EXCLUDED.wind_v,
            precip_3h = EXCLUDED.precip_3h,
            temp_k    = EXCLUDED.temp_k,
            fetched_at = NOW()
    """
    execute_values(cur, sql, records)
    conn.commit()
    cur.close()
    conn.close()
    print(f"✅ Windy insert done — {len(records)} rows")

insert_windy(windy_df, DB_CONFIG)

## 5. Train Isolation Forest on ERA5

In [ ]:
FEATURES = ["wind_speed", "precip_3h", "temp_k"]

X_train = era5_df[FEATURES].dropna().values

scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X_train)

iso_forest = IsolationForest(
    n_estimators=IF_N_ESTIMATORS,
    contamination=IF_CONTAMINATION,
    random_state=42,
    n_jobs=-1,
)
iso_forest.fit(X_scaled)

joblib.dump(iso_forest, MODEL_PATH)
joblib.dump(scaler,     SCALER_PATH)
print(f"✅ Model trained on {len(X_train):,} ERA5 samples")
print(f"   Saved → {MODEL_PATH}, {SCALER_PATH}")

In [ ]:
# Score threshold used by the model (decision_function == 0 boundary)
threshold = iso_forest.offset_
print(f"Decision threshold (offset_): {threshold:.4f}")

# In-sample anomaly rate sanity check
preds = iso_forest.predict(X_scaled)
anomaly_rate = (preds == -1).mean()
print(f"In-sample anomaly rate: {anomaly_rate:.2%}  (target: {IF_CONTAMINATION:.2%})")

## 6. Inference — Score Windy Forecasts

In [ ]:
def run_inference(df: pd.DataFrame, source: str) -> pd.DataFrame:
    """
    Runs Isolation Forest on any DataFrame that has wind_speed, precip_3h, temp_k.
    Returns the input DataFrame with anomaly_score and is_anomaly columns appended.
    """
    model  = joblib.load(MODEL_PATH)
    sc     = joblib.load(SCALER_PATH)

    out = df.copy()
    valid = out[FEATURES].dropna()

    X      = sc.transform(valid.values)
    scores = model.decision_function(X)   # higher = more normal
    labels = model.predict(X)             # 1 = normal, -1 = anomaly

    out.loc[valid.index, "anomaly_score"] = scores
    out.loc[valid.index, "is_anomaly"]    = labels == -1
    out["source"]    = source
    out["threshold"] = model.offset_

    return out


windy_scored = run_inference(windy_df, source="windy")

n_anomalies = windy_scored["is_anomaly"].sum()
print(f"Windy anomalies detected: {n_anomalies} / {len(windy_scored)} ({n_anomalies/len(windy_scored):.1%})")
windy_scored[windy_scored["is_anomaly"]][["time","lat","lon","wind_speed","precip_3h","temp_k","anomaly_score"]].head(10)

## 7. Store Anomaly Scores

In [ ]:
def insert_scores(df: pd.DataFrame, config: dict):
    conn = psycopg2.connect(**config)
    cur  = conn.cursor()

    scored = df.dropna(subset=["anomaly_score"])
    records = [
        (
            row.time, row.lat, row.lon, row.source,
            row.anomaly_score, bool(row.is_anomaly), row.threshold,
            row.wind_speed, row.precip_3h, row.temp_k,
        )
        for row in scored.itertuples()
    ]

    sql = """
        INSERT INTO anomaly_scores
            (time, lat, lon, source, anomaly_score, is_anomaly, threshold, wind_speed, precip_3h, temp_k)
        VALUES %s
        ON CONFLICT (time, lat, lon, source) DO UPDATE SET
            anomaly_score = EXCLUDED.anomaly_score,
            is_anomaly    = EXCLUDED.is_anomaly,
            threshold     = EXCLUDED.threshold
    """
    execute_values(cur, sql, records)
    conn.commit()
    cur.close()
    conn.close()
    print(f"✅ Scores stored — {len(records)} rows")


insert_scores(windy_scored, DB_CONFIG)

## 8. Quick Visualisation

In [ ]:
CITY_NAMES = {
    (51.18, 71.45): "Astana",
    (43.25, 76.95): "Almaty",
    (42.30, 69.60): "Shymkent",
}

for point in GRID_POINTS:
    sub = windy_scored[
        (windy_scored["lat"] == point["lat"]) &
        (windy_scored["lon"] == point["lon"])
    ].copy().sort_values("time")
    if sub.empty:
        continue

    city = CITY_NAMES.get((point["lat"], point["lon"]), f"lat={point['lat']} lon={point['lon']}")
    date_range = f"{sub['time'].min().strftime('%b %d %Y')} – {sub['time'].max().strftime('%b %d %Y')}"

    fig, axes = plt.subplots(3, 1, figsize=(16, 8), sharex=True)
    fig.suptitle(f"{city} — Weather Anomaly Detection\n{date_range}", fontsize=14, fontweight="bold")

    var_cfg = [
        ("wind_speed", "Wind Speed (m/s)",         "steelblue",  None),
        ("precip_3h",  "Precipitation 3h (mm)",     "teal",       1000),  # m → mm
        ("temp_k",     "Temperature (°C)",           "tomato",     None),  # K → °C
    ]

    for ax, (col, label, color, scale) in zip(axes, var_cfg):
        y = sub[col].copy()
        if scale:
            y = y * scale
        if col == "temp_k":
            y = y - 273.15

        ax.plot(sub["time"], y, color=color, lw=1.2, alpha=0.85)

        anomalies = sub[sub["is_anomaly"] == True]
        ya = anomalies[col].copy()
        if scale:
            ya = ya * scale
        if col == "temp_k":
            ya = ya - 273.15

        ax.scatter(anomalies["time"], ya, color="red", zorder=5, s=45,
                   label=f"Anomaly (n={len(anomalies)})")
        ax.set_ylabel(label, fontsize=10)
        ax.legend(loc="upper right", fontsize=9)
        ax.grid(True, alpha=0.3)

        # year markers
        for year in sub["time"].dt.year.unique():
            yr_start = pd.Timestamp(f"{year}-01-01", tz="UTC")
            if sub["time"].min() <= yr_start <= sub["time"].max():
                ax.axvline(yr_start, color="gray", lw=1, linestyle="--", alpha=0.6)
                ax.text(yr_start, ax.get_ylim()[1], str(year),
                        fontsize=8, color="gray", ha="left", va="top")

    axes[-1].xaxis.set_major_locator(mdates.DayLocator(interval=2))
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d\n%Y"))
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Thresholds from ERA5 training data (per city nearest grid point) ──────────
TEMP_ANOMALY_PERCENTILE  = 1   # below 1st percentile = cold anomaly
WIND_ANOMALY_PERCENTILE  = 99  # above 99th = wind anomaly
PRECIP_ANOMALY_PERCENTILE = 99 # above 99th = precip anomaly

def compute_thresholds(df: pd.DataFrame) -> dict:
    return {
        "temp_low":    np.percentile(df["temp_k"].dropna(),    TEMP_ANOMALY_PERCENTILE),
        "wind_high":   np.percentile(df["wind_speed"].dropna(), WIND_ANOMALY_PERCENTILE),
        "precip_high": np.percentile(df["precip_3h"].dropna(), PRECIP_ANOMALY_PERCENTILE),
    }

thresholds = compute_thresholds(era5_df)
print(f"Temp   < {thresholds['temp_low'] - 273.15:.1f}°C  → cold anomaly")
print(f"Wind   > {thresholds['wind_high']:.1f} m/s → wind anomaly")
print(f"Precip > {thresholds['precip_high']*1000:.2f} mm → precip anomaly")


def run_inference_hybrid(df: pd.DataFrame, source: str) -> pd.DataFrame:
    model  = joblib.load(MODEL_PATH)
    sc     = joblib.load(SCALER_PATH)

    out   = df.copy()
    valid = out[FEATURES].dropna()
    X     = sc.transform(valid.values)

    scores = model.decision_function(X)
    if_flag = model.predict(X) == -1

    # univariate hard rules
    rule_flag = (
        (valid["temp_k"]     < thresholds["temp_low"])    |
        (valid["wind_speed"] > thresholds["wind_high"])   |
        (valid["precip_3h"]  > thresholds["precip_high"])
    )

    # anomaly if EITHER IF or any rule fires
    combined = if_flag | rule_flag.values

    out.loc[valid.index, "anomaly_score"] = scores
    out.loc[valid.index, "is_anomaly"]    = combined
    out.loc[valid.index, "if_flag"]       = if_flag
    out.loc[valid.index, "rule_flag"]     = rule_flag.values
    out["source"]    = source
    out["threshold"] = model.offset_

    return out


windy_scored = run_inference_hybrid(windy_df, source="windy")

# show what each flag caught
print(f"\nIF only:        {(windy_scored['if_flag'] & ~windy_scored['rule_flag']).sum()}")
print(f"Rule only:      {(windy_scored['rule_flag'] & ~windy_scored['if_flag']).sum()}")
print(f"Both:           {(windy_scored['if_flag'] & windy_scored['rule_flag']).sum()}")
print(f"Total anomalies:{windy_scored['is_anomaly'].sum()}")